# A full business solution


### BUSINESS CHALLENGE:

Creating a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://www.infinite.com")
links

['#content',
 '/',
 '/#industries',
 'https://www.infinite.com/healthcare/',
 'https://www.infinite.com/banking-and-financial-services/',
 'https://www.infinite.com/insurance/',
 'https://www.infinite.com/telecom/',
 'https://www.infinite.com/life-sciences/',
 'https://www.infinite.com/high-technology/',
 'https://www.infinite.com/media-entertainment/',
 'https://www.infinite.com/government/',
 'https://www.infinite.com/manufacturing/',
 '/#services',
 'https://www.infinite.com/application-services/',
 'https://www.infinite.com/ai/',
 'https://www.infinite.com/cloud-infrastructure/',
 'https://www.infinite.com/cybersecurity/',
 'https://www.infinite.com/data-analytics/',
 'https://www.infinite.com/digital-engineering-services/',
 'https://www.infinite.com/intelligent-automation/',
 'https://www.infinite.com/quality-engineering/',
 'https://www.infinite.com/brassring-solutions/',
 'https://www.infinite.com/small-business-solutions/',
 'https://www.infinite.com/infinite-convergence/',
 '

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://infinte.com"))

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.



Here is the list of links on the website https://infinte.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):




In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://infinte.com")

{'links': []}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://infinite.com")

Selecting relevant links for https://infinite.com by calling gpt-5-nano
Found 36 relevant links


{'links': [{'type': 'about page', 'url': 'https://www.infinite.com/about-us/'},
  {'type': 'alliances/partners page',
   'url': 'https://www.infinite.com/alliances-partners/'},
  {'type': 'awards/recognition page',
   'url': 'https://www.infinite.com/awards-recognition/'},
  {'type': 'news and events page',
   'url': 'https://www.infinite.com/news-and-events/'},
  {'type': 'resources page', 'url': 'https://www.infinite.com/resources/'},
  {'type': 'careers page', 'url': 'https://www.infinite.com/careers/'},
  {'type': 'contact page', 'url': 'https://www.infinite.com/contact/'},
  {'type': 'homepage', 'url': 'https://www.infinite.com/'},
  {'type': 'healthcare industry page',
   'url': 'https://www.infinite.com/healthcare/'},
  {'type': 'banking and financial services industry page',
   'url': 'https://www.infinite.com/banking-and-financial-services/'},
  {'type': 'insurance industry page',
   'url': 'https://www.infinite.com/insurance/'},
  {'type': 'telecom industry page',
   'url': '

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Endpoints product', 'url': 'https://endpoints.huggingface.co'}]}



Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-M2.7
Updated
about 24 hours ago
•
143k
•
840
tencent/HY-Embodied-0.5
Updated
2 days ago
•
1.06k
•
768
zai-org/GLM-5.1
Updated
about 13 hours ago
•
94.4k
•
1.27k
google/gemma-4-31B-it
Updated
6 days ago
•
3.2M
•
1.97k
baidu/ERNIE-Image
Updated
about 12 hours ago
•
1.35k
•
368
Browse 2M+ models
Spaces
Running
on
Zero
Featured
485
OmniVoice
🌍
485
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.06k
Wan2.2 14B Preview
🐌
2.06k
generate a video from an image with a text prompt
Running
on
A10G
Featu

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""




In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("HuggingFace", "https://infinite.com")

Selecting relevant links for https://infinite.com by calling gpt-5-nano
Found 33 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nInfinite | Global Digital Engineering & IT Services\n\nSkip to content\nIndustries\nHealthcare\nBanking & Financial Services\nInsurance\nTelecom\nLife Sciences\nHigh Technology\nMedia & Entertainment\nGovernment\nManufacturing\nProviding cutting-edge digital experiences across industries\nWe partner with clients in healthcare, finance, telecom, government, and more to create innovative solutions that tackle their specific challenges and foster growth.\nServices\nApplication Services\nArtificial Intelligence\nCloud & Infrastructure\nCybersecurity\nData & Analytics\nDigital Engineering Services\nIntelligent Automation\nQuality Engineering\nNext-Generation Services\nEmpowering businesses with digital solutions like cloud, AI, and automation to improve

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://infinite.com")

Selecting relevant links for https://infinite.com by calling gpt-5-nano
Found 43 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Infinite Computer Solutions: Your Global Digital Engineering & IT Partner

---

## Company Overview

Infinite Computer Solutions is a global leader in digital engineering and IT services with over 20 years of experience driving digital transformation across industries. Our mission is to empower businesses with innovative digital solutions such as cloud computing, artificial intelligence, automation, and cybersecurity to optimize operations, accelerate growth, and stay agile in the rapidly evolving technology landscape.

Through strategic partnerships and a focus on delivering measurable business outcomes, Infinite partners with clients worldwide to build future-ready enterprises "tomorrow, today."

---

## Industries Served

We deliver cutting-edge digital experiences tailored to the specific needs and challenges of diverse industries:

- Healthcare
- Banking & Financial Services
- Insurance
- Telecom
- Life Sciences
- High Technology
- Media & Entertainment
- Government
- Manufacturing

---

## Services & Solutions

### Services
- **Application Services:** Development and management of enterprise applications.
- **Artificial Intelligence:** AI-powered solutions to drive innovation and efficiency.
- **Cloud & Infrastructure:** Scalable cloud solutions ensuring business continuity.
- **Cybersecurity:** Protecting enterprise assets and data in a connected world.
- **Data & Analytics:** Turning data into actionable business insights.
- **Digital Engineering Services:** Engineering next-gen digital products and platforms.
- **Intelligent Automation:** Automating processes to enhance productivity.
- **Quality Engineering:** Ensuring superior product quality and user experience.
- **Next-Generation Services:** Advanced technology solutions for future-proofing business.

### Proprietary Solutions
- **BrassRing:** Talent acquisition and onboarding platform streamlining recruitment.
- **Turbify:** All-in-one website hosting and e-commerce solution for businesses.
- **NetSfere:** Secure enterprise messaging platform ensuring confidential communications.
- **Sensyon:** Integration platform for intelligent Internet of Things (IoT).

These solutions enable streamlined recruitment, enhanced online business capabilities, secure enterprise communication, and intelligent connected environments.

---

## Company Culture & Values

At Infinite, our culture centers on innovation, collaboration, and client-centricity. We invest deeply in understanding our clients' business context to deliver technology solutions that produce real-world results. Our teams are empowered to drive continuous improvement through creativity and expertise, fostering an environment where talent thrives through learning and growth.

We also prioritize corporate social responsibility, sustainability, and actively engage in partnerships and sponsorships that amplify our positive impact on the communities we serve.

---

## Careers & Opportunities

Infinite offers dynamic career opportunities for professionals eager to work at the forefront of digital transformation. With a global presence and diverse industry exposure, employees enjoy:

- Innovative and challenging projects
- Collaborative work culture
- Continuous learning and professional development
- Opportunities across multiple technology disciplines including AI, cloud, cybersecurity, and data analytics
- A commitment to work-life balance and sustainability

---

## Why Partner with Infinite?

- Over 20 years of expertise in technology innovation and business transformation
- Global reach with deep domain knowledge across key industries
- End-to-end digital engineering and IT services tailored to your business needs
- Proven proprietary solutions enhancing operational efficiency and security
- Commitment to delivering measurable business outcomes and client success

---

## Contact and Global Presence

Infinite Computer Solutions operates globally with multiple locations serving clients around the world. To learn more about our offerings, request a consultation, or explore career opportunities, please visit our website or contact us directly.

**Digitally Engineering Tomorrow, Today.**

---

*Infinite Computer Solutions – Driving Digital Transformation with Expertise and Innovation.*

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://infinite.com")

Selecting relevant links for https://infinite.com by calling gpt-5-nano
Found 12 relevant links


# Infinite Computer Solutions Brochure

---

## Company Overview

**Infinite Computer Solutions** is a global leader in digital engineering and IT services with over 20 years of experience driving digital transformation. The company specializes in empowering businesses to optimize and scale their technology by leveraging innovative solutions, industry expertise, and a strong focus on achieving measurable business outcomes. Infinite partners with clients across various industries to create cutting-edge digital experiences that address specific challenges and foster growth.

---

## Industries Served

- Healthcare  
- Banking & Financial Services  
- Insurance  
- Telecom  
- Life Sciences  
- High Technology  
- Media & Entertainment  
- Government  
- Manufacturing  

Infinite acts as a trusted partner by providing tailored solutions that meet the unique needs of these diverse sectors.

---

## Services

Infinite offers a comprehensive portfolio of IT and engineering services:

- **Application Services**  
- **Artificial Intelligence**  
- **Cloud & Infrastructure**  
- **Cybersecurity**  
- **Data & Analytics**  
- **Digital Engineering Services**  
- **Intelligent Automation**  
- **Quality Engineering**  
- **Next-Generation Services**  

These services help businesses improve efficiency, drive growth, and remain agile in a fast-changing digital world.

---

## Innovative Solutions

Infinite delivers a range of innovative, business-focused solutions designed to solve critical challenges:

- **BrassRing**: Talent acquisition and onboarding solution streamlining recruitment processes.  
- **Turbify**: All-in-one website hosting and e-commerce platform empowering small businesses online.  
- **NetSfere**: Secure enterprise messaging solutions to ensure robust communication.  
- **Sensyon**: Integration platform for intelligent Internet of Things (IoT) applications enabling seamless digital transformation.

---

## Company Culture

Infinite values innovation, strategic partnership, and a client-centric approach. The company fosters a culture focused on:

- Collaborating closely with clients to deeply understand their business.  
- Driving continuous innovation through emerging technologies like AI, cloud, and automation.  
- Encouraging agility and responsiveness to rapidly evolving market needs.  
- Committing to corporate social responsibility and sustainability initiatives.  

Infinite's dedication to these principles nurtures an environment where employees thrive and clients succeed.

---

## Careers and Opportunities

Infinite offers rewarding career paths across various domains in digital engineering and IT services. The company is dedicated to:

- Recruiting diverse talent passionate about technology and innovation.  
- Providing growth and learning opportunities to help employees advance professionally.  
- Creating an inclusive workplace that encourages collaboration and creativity.  

Prospective candidates can join Infinite to be part of a forward-thinking organization shaping the future of technology.

---

## Contact and Locations

Infinite Computer Solutions operates globally, supporting clients wherever their digital transformation journey takes them. For more information or to explore partnership opportunities:

- Visit the company website  
- Explore career openings  
- Contact the sales or support teams  

---

## Summary

Infinite Computer Solutions is your partner for **digitally engineering tomorrow, today**. With expertise spanning multiple industries, a broad services portfolio, and innovative solutions, Infinite helps organizations meet their unique challenges and accelerate growth in an increasingly digital world. Whether you are a prospective customer, investor, or career-seeker, Infinite stands ready to collaborate on the future of digital technology.

---

*Driving Digital Transformation with Expertise and Innovation*